In [5]:
# %pip install numpy matplotlib imageio pillow

In [ ]:

nodes = ['A', 'P', 'XVIII', 'XI', '4', '5', '6', '7', '9', '11']
destinations = ['4', '5', '6', '7', '9', '11']

adj = {
    'A': ['XVIII'],
    'XVIII': ['5', '6', 'XI'],
    '6': ['7'],
    'P': ['XI'],
    'XI': ['9', '4', '11', 'X'],
    # ...
}
capacities = {
    ('P', 'XI'): 2670.3,
    ('XVIII', 'XI'): 3644.6,
    ('XVIII', '6'): 4878.2,
    # ...
}

requests = {
    ('A', '1'): 738.0
    # ...
}



---

# Оптимизация распределенной электрической сети методом Муравьиной колонии (ACS)

### Описание логики задачи
Данная реализация адаптирует классический муравьиный алгоритм под задачу распределения мощностей в электрической сети «Альфа».

**Основные концепции адаптации:**
1.  **Квантование энергии (Муравьи-киловатты):** Каждая заявка на поставку (например, 738 кВт) разбивается на мелкие порции — **кванты** (по умолчанию 10 кВт). Один муравей в нашей системе — это один такой квант.
2.  **Равнодоступность:** Все муравьи от всех поставщиков перемешиваются в общем пуле (`ants_pool`). Это гарантирует, что при возникновении дефицита мощности на участках, объемы будут ограничиваться пропорционально и честно, так как муравьи конкурируют за «место в проводе» одновременно.
3.  **3D-Феромоны (Навигация по целям):** В отличие от стандартных задач, здесь муравьи имеют разные цели (разных потребителей). Поэтому матрица феромонов имеет три измерения: `[откуда][куда][целевой_потребитель]`. Это позволяет муравьям не путаться в чужих следах и строить маршруты именно к своему потребителю.
4.  **Динамическая эвристика (Свободная мощность):** Вместо расстояния муравьи оценивают **доступную пропускную способность** участка. Если линия начинает «забиваться», её привлекательность для муравьев падает, что заставляет их искать обходные пути еще до того, как лимит будет полностью исчерпан.

---

### Ключевые моменты проделанной работы:
1.  **Защита от перегрузок:** Условие `if free_cap >= quantum` гарантирует, что ни один муравей не нарушит технические ограничения, указанные в Таблице 1.2.
2.  **Интеллектуальный обход:** Благодаря параметру `beta`, муравьи будут «чувствовать» снижение свободной мощности на популярных участках и автоматически выбирать альтернативные пути из словаря `adj`.
3.  **Изоляция потоков:** Использование `pheromones[u][v][target]` позволяет моделировать сложную сеть, где потоки к разным потребителям не мешают друг другу в плане навигации, но конкурируют за физический ресурс — мощность линии.

In [4]:
import numpy as np
import random

def ant_colony_power_flow(nodes, destinations, adj, capacities, requests,
                          quantum=10.0, alpha=1.0, beta=3.0, rho=0.3,
                          Q=100.0, n_iterations=20, tau0=1.0, seed=337):
    """
    Алгоритм муравьиной колонии для распределения потоков в электросети.

    Аргументы:
        nodes: Список всех узлов сети (включая источники и потребители).
        destinations: Список узлов-потребителей.
        adj: Словарь топологии {узел: [список_соседей]} по Схеме 1.
        capacities: Словарь лимитов {(узел_А, узел_Б): мощность_кВт} по Таблице 1.2.
        requests: Словарь заявок {(источник, потребитель): объем_кВт} по Таблице 1.1.
        quantum: Вес одного муравья в кВт (точность дробления потока).
        alpha: Вес накопленного опыта (феромона).
        beta: Вес текущего состояния (свободной мощности).
        rho: Коэффициент испарения феромона.
        Q: Константа для награждения за успешный путь.
        n_iterations: Количество циклов перераспределения (поколений).
        tau0: Начальный уровень "привлекательности" всех дорог.
        seed: Зерно случайных чисел для воспроизводимости.
    """

    # Константа "бесконечной" мощности для участков, не указанных в Таблице 1.2
    INF = 1_000_000_000.0
    rng = np.random.RandomState(seed)

    # 1. Инициализация 3D матрицы феромонов
    # Структура: [Текущий_Узел][Следующий_Узел][Конечный_Потребитель]
    pheromones = {
        u: {v: {dest: tau0 for dest in destinations} for v in neighbors}
        for u, neighbors in adj.items()
    }

    history = []

    # ГЛАВНЫЙ ЦИКЛ ПОКОЛЕНИЙ (ИТЕРАЦИИ ОБУЧЕНИЯ)
    for iteration in range(n_iterations):

        # Обнуляем текущую загрузку сети в начале каждого поколения
        # current_load хранит количество кВт, уже занятых на каждом участке
        current_load = {edge: 0.0 for edge in capacities.keys()}

        # 2. Формирование пула муравьев (Квантование заявок)
        ants_pool = []
        for (src, dst), volume in requests.items():
            # Рассчитываем, сколько муравьев нужно для этой заявки
            num_ants_for_req = int(volume / quantum)
            for _ in range(num_ants_for_req):
                ants_pool.append((src, dst))

        # Перемешивание пула обеспечивает "равнодоступность" поставщиков
        # Муравьи от разных источников будут бороться за узкие места вперемешку
        random.shuffle(ants_pool)

        # 3. ДВИЖЕНИЕ МУРАВЬЕВ
        for ant in ants_pool:
            current_node, target = ant  # Извлекаем точку старта и цель муравья
            path = [current_node]
            visited = {current_node}    # Защита от зацикливания (не ходить кругами)

            # Муравей движется по графу, пока не достигнет цели
            while current_node != target:
                neighbors = adj.get(current_node, [])

                if not neighbors:
                    break  # Тупик на схеме

                attractions = []
                valid_neighbors = []

                # ОЦЕНКА ДОСТУПНЫХ ВАРИАНТОВ ПЕРЕХОДА
                for v in neighbors:
                    if v in visited:
                        continue # Пропускаем узлы, где уже были в этом рейсе

                    edge = (current_node, v)

                    # Получаем лимит мощности участка (из таблицы или бесконечность)
                    limit = capacities.get(edge, INF)

                    # Проверяем свободное место: общий лимит минус уже занятое другими муравьями
                    free_cap = limit - current_load.get(edge, 0.0)

                    # Если квант энергии физически может пройти (место есть)
                    if free_cap >= quantum:
                        # Извлекаем феромон именно для нашей цели (target)
                        tau_val = pheromones[current_node][v][target]

                        # ФОРМУЛА ПРИВЛЕКАТЕЛЬНОСТИ (Вероятностное правило перехода)
                        # (Опыт других муравьев) ^ alpha * (Свободное место) ^ beta
                        # min(free_cap, 10000.0) используется для стабилизации вычислений
                        attr = (tau_val ** alpha) * (min(free_cap, 10000.0) ** beta)

                        attractions.append(attr)
                        valid_neighbors.append(v)

                # Если путей нет (все забито или тупик), муравей прекращает движение
                if not attractions:
                    break

                # --- ЗДЕСЬ БУДЕТ ВЫБОР СЛЕДУЮЩЕГО УЗЛА И ОБНОВЛЕНИЕ СОСТОЯНИЯ ---
                # (Логика будет продолжена в следующем блоке реализации)
                break # Временная заглушка для текущего этапа разработки

    return history